For licensing see accompanying LICENSE file.  
Copyright (C) 2025 Apple Inc. All Rights Reserved.

# Metric Testing
Test metric scores against Neuronpedia.

In [3]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Change working directory to project root
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

In [ ]:
import metrics
import features

In [ ]:
eval_model_name = 'gpt-4o'
data_model_name = 'gpt-4o'

## Testing Eleuther Metrics Against Neuronpedia Results

### Detection

In [21]:
# 100% Detection on Neuronpedia. This should also be high.
feature = features.Feature(
    model_id="gemma-2-9b",
    layer="31-gemmascope-res-16k",
    index=16345
)
description = "information related to cellular responses and treatments"
detection_metric = metrics.Detection(
    eval_model_name=eval_model_name,
    data_model_name=data_model_name,
    is_semantic_regex=False,
    ignore_first_token=False,
    show_breaks=True,
)
detection_result = detection_metric.compute(
    description,
    feature,
    logging=True);

DETECTION MATCH PROMPT:
SYSTEM: You are an intelligent and meticulous linguistics researcher.

You will be given a certain latent of text, such as "male pronouns" or "text with negative sentiment".

You will then be given several text examples. Your task is to determine which examples possess the latent.

For each example in turn, return 1 if the sentence is correctly labeled or 0 if the tokens are mislabeled. You must return your response in a valid Python list. Do not return anything else besides a Python list.


USER: Latent explanation: Words related to American football positions, specifically the tight end position.

Test examples:

Example 0:<|endoftext|>Getty Images

Patriots tight end Rob Gronkowski had his boss'
Example 1: names of months used in The Lord of the Rings:

"...the
Example 2: Media Day 2015

LSU defensive end Isaiah Washington (94) speaks to the
Example 3: shown, is generally not eligible for ads. For example, videos about recent tragedies,
Example 4: line, with 

In [22]:
# 0% Detection on Neuronpedia. 0% is an artifact because they don't have any negatives, but this should also be low.
feature = features.Feature(
    model_id="gemma-2-9b",
    layer="31-gemmascope-res-16k",
    index=15964
)
description = "medical terminology and technical terms related to cardiac and clinical conditions."
detection_metric = metrics.Detection(
    eval_model_name=eval_model_name,
    data_model_name=data_model_name,
    is_semantic_regex=False,
    ignore_first_token=False,
    show_breaks=True,
)
detection_result = detection_metric.compute(
    description,
    feature,
    logging=True);

DETECTION MATCH PROMPT:
SYSTEM: You are an intelligent and meticulous linguistics researcher.

You will be given a certain latent of text, such as "male pronouns" or "text with negative sentiment".

You will then be given several text examples. Your task is to determine which examples possess the latent.

For each example in turn, return 1 if the sentence is correctly labeled or 0 if the tokens are mislabeled. You must return your response in a valid Python list. Do not return anything else besides a Python list.


USER: Latent explanation: Words related to American football positions, specifically the tight end position.

Test examples:

Example 0:<|endoftext|>Getty Images

Patriots tight end Rob Gronkowski had his boss'
Example 1: names of months used in The Lord of the Rings:

"...the
Example 2: Media Day 2015

LSU defensive end Isaiah Washington (94) speaks to the
Example 3: shown, is generally not eligible for ads. For example, videos about recent tragedies,
Example 4: line, with 

### Fuzzing


In [25]:
# 100% Detection on Neuronpedia. This should also be high.
feature = features.Feature(
    model_id="gemma-2-2b",
    layer="20-gemmascope-res-16k",
    index=13
)
description = "miss"
fuzzing_metric = metrics.Fuzzing(
    eval_model_name=eval_model_name,
    data_model_name=data_model_name,
    is_semantic_regex=False,
    ignore_first_token=False,
    show_breaks=True,
)
fuzzing_result = fuzzing_metric.compute(
    description,
    feature,
    logging=True);

FUZZING - sampled 30 positive and 30 negative examples
FUZZING MATCH PROMPT:
SYSTEM: You are an intelligent and meticulous linguistics researcher.

You will be given a certain latent of text, such as "male pronouns" or "text with negative sentiment".

You will be given a few examples of text that contain this latent. Portions of the sentence which strongly represent this latent are between tokens << and >>.

Some examples might be mislabeled. Your task is to determine if every single token within << and >> is correctly labeled. Consider that all provided examples could be correct, none of the examples could be correct, or a mix. An example is only correct if every marked token is representative of the latent

For each example in turn, return 1 if the sentence is correctly labeled or 0 if the tokens are mislabeled. You must return your response in a valid Python list. Do not return anything else besides a Python list.


USER: Latent explanation: Words related to American football positi

## Testing Against FADE Results

In [19]:
fade_metrics = [
    metrics.Clarity(eval_model_name, data_model_name, False, False, True),
    metrics.Responsiveness(eval_model_name, data_model_name, False, False, True),
    metrics.Purity(eval_model_name, data_model_name, False, False, True),
]

In [20]:
# FADE paper shows low scores for Neuronpedia description. Our FADE scores should also be low.
feature = features.Feature(
    model_id="gemma-2-2b",
    layer="20-gemmascope-res-16k",
    index=9295
)
description = "the presence of JavaScript code segments or functions"
for fade_metric in fade_metrics:
    fade_result = fade_metric.compute(
        description,
        feature,
        logging=False);

CLARITY - computing for feature gemma-2-2b_20-gemmascope-res-16k_9295 with baseline description: the presence of JavaScript code segments or functions
CLARITY - 50 generated positive: [(0, '<bos>const numbers = [1, 2, 3, 4, 5];'), (0, '<bos>const sum = (a, b) => a + b;'), (0, "<bos>function greet() { console.log('Hello, World!'); }"), (0, '<bos>let x = 10; console.log(`The value of x is ${x}`);'), (0, '<bos>let x = 10; const double = x * 2; console.log(double);'), (0, "<bos>document.getElementById('myButton').addEventListener('click', function() { alert('Button clicked!'); });"), (0, "<bos>document.getElementById('myElement').innerHTML = 'Hello World!';"), (0, '<bos>let add = (a, b) => a + b;'), (0, "<bos>for (let i = 0; i < 5; i++) { console.log('Count: ' + i); }"), (0, '<bos>let number = 10; console.log(`The number is ${number}`);'), (0, '<bos>async function fetchData(url) { const response = await fetch(url); return response.json(); }'), (0, "<bos>if (user.isLoggedIn) { console.log('

In [21]:
# FADE paper shows hgiher scores for Neuronpedia description. Our FADE scores should also be higher.
feature = features.Feature(
    model_id="gemma-2-2b",
    layer="20-gemmascope-res-16k",
    index=1139
)
description = "references to problematic situations or conflicts that cause trouble"
for fade_metric in fade_metrics:
    fade_result = fade_metric.compute(
        description,
        feature,
        logging=False);

CLARITY - computing for feature gemma-2-2b_20-gemmascope-res-16k_1139 with baseline description: references to problematic situations or conflicts that cause trouble
CLARITY - 50 generated positive: [(17.3125, "<bos>It's like walking on a tightrope, one wrong step and chaos ensues."), (0, '<bos>It’s always the little things that tip the scale, like a whisper in a crowded room.'), (0, '<bos>In the pursuit of success, friendships turned into battlegrounds.'), (0, '<bos>As tensions rose, the room felt charged, a powder keg waiting to ignite.'), (10.09375, '<bos>Caught between a rock and a hard place, the choice is never easy.'), (0, '<bos>Every step forward feels like two steps back when childhood friends become rivals.'), (8.8203125, '<bos>In the game of chess, sometimes the best move is to avoid confrontation altogether.'), (0, "<bos>It's like trying to untangle a web of lies, where every pull only tightens the knot."), (73.25, '<bos>He found himself in hot water after the controversial